In [ ]:
from collections import deque
import threading
import cv2
import joblib
import numpy as np
import serial
from MODEL_USING import *
from PreProcessing import imgPreProcessing, numPreProcessing
from ultralytics import YOLO

# ==================== CẤU HÌNH SERIAL & MODEL ====================
SERIAL_PORT = "COM3"  # Thay đúng cổng COM ESP32/Arduino
BAUD_RATE = 115200

path_model_yolo = r"C:\Users\trann\Documents\NHAT_NAM_TRAN\WORK_SPACE\NGHIEN_CUU_KHOA_HOC\NCKH_MODEL_AI\BAO_CAO\TONG HOP KET QUA\yolo11l_40kimg\best.pt"
path_model_rf = "rf_fire_smoke_model.pkl"

# Nạp các model
detection_model = MODEL(path_model_yolo)
rf_model = joblib.load(path_model_rf)
class_names = {0: "Person", 1: "Fire", 2: "Smoke"}

# Biến chia sẻ trạng thái từ Serial
sensor_state = {
    "pred_label": 0,
    "confidence": 0.0,
    "temp": 0.0,
    "hum": 0,
    "gas": 0,
}
buf_gas = deque(maxlen=10)
is_running = True


# ==================== LUỒNG ĐỌC SERIAL NGẦM ====================
def serial_reader_thread():
    global is_running, sensor_state
    try:
        ser = serial.Serial(SERIAL_PORT, BAUD_RATE, timeout=0.1)
    except Exception as e:
        print(f"[Serial Error] Không mở được cổng: {e}")
        return

    while is_running:
        if ser.in_waiting:
            line = ser.readline().decode("utf-8", errors="ignore").strip()
            # Giả định ESP32 gửi định dạng chuỗi CSV: "nhiet_do,do_am,khi_gas"
            parts = line.split(",")
            if len(parts) == 3:
                try:
                    temp = float(parts[0])
                    hum = float(parts[1])
                    gas = float(parts[2])

                    # Rolling Mean 10 cho Gas
                    buf_gas.append(gas)
                    gas_smooth = sum(buf_gas) / len(buf_gas)

                    # Dự đoán qua Random Forest
                    features = np.array([[temp, hum, gas_smooth]])
                    rf_probs = rf_model.predict_proba(features)[0]
                    pred_label = np.argmax(rf_probs)
                    conf = rf_probs[pred_label]

                    # Cập nhật kết quả
                    sensor_state["pred_label"] = int(pred_label)
                    sensor_state["confidence"] = float(conf)
                    sensor_state["temp"] = temp
                    sensor_state["hum"] = hum
                    sensor_state["gas"] = gas_smooth
                except ValueError:
                    pass
    ser.close()


# Khởi chạy Thread Serial
thread = threading.Thread(target=serial_reader_thread, daemon=True)
thread.start()

# ==================== VÒNG LẶP XỬ LÝ CAMERA ====================
cap = cv2.VideoCapture(0)
MAX_PATIENCE = 10
patience_counters = {}
saved_boxes = {}

while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    clahe_img = imgPreProcessing.clahe_img_ret(frame, 1, (8, 8))
    detected_classes_this_frame = set()
    boxes = detection_model.boxes(clahe_img)

    for box in boxes:
        cls, conf_img, (x1, y1, x2, y2) = MODEL.get_box_info(
            detection_model, box
        )
        detected_classes_this_frame.add(cls)
        saved_boxes[cls] = (x1, y1, x2, y2, conf_img)
        patience_counters[cls] = MAX_PATIENCE

    # Lấy thông tin hiện tại từ Serial
    rf_label = sensor_state["pred_label"]
    rf_conf = sensor_state["confidence"]

    for cls in list(patience_counters.keys()):
        if cls not in detected_classes_this_frame:
            patience_counters[cls] -= 1

        if patience_counters[cls] > 0:
            x1, y1, x2, y2, conf_img = saved_boxes[cls]
            label_name = class_names.get(cls, f"Unknown {cls}")

            # ĐIỀU KIỆN KÍCH HOẠT KÉP: Cả YOLO và Serial RF đều > 0.5 và khớp trạng thái Lửa/Khói
            is_fused_alert = False
            if (cls in [1, 2]) and (rf_label == cls):
                if conf_img > 0.5 and rf_conf > 0.5:
                    is_fused_alert = True
                    print(
                        f"[CẢNH BÁO XÁC THỰC KÉP] {label_name.upper()} | YOLO Conf: {conf_img:.2f} | Sensor Conf: {rf_conf:.2f}"
                    )

            # Màu sắc: Đỏ đậm nếu kích hoạt kép, Vàng nếu chỉ có YOLO đơn lẻ
            box_color = (
                (0, 0, 255)
                if is_fused_alert
                else ((0, 255, 255) if cls in [1, 2] else (255, 0, 0))
            )
            cv2.rectangle(frame, (x1, y1), (x2, y2), box_color, 2)

            text = f"{label_name} (YOLO:{conf_img:.2f} | RF:{rf_conf:.2f})"
            cv2.putText(
                frame,
                text,
                (x1, y1 - 10),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.55,
                box_color,
                2,
            )
        else:
            patience_counters.pop(cls, None)
            saved_boxes.pop(cls, None)

    # Hiển thị thông số Serial lên góc trên màn hình
    info_text = f"Sensor: T={sensor_state['temp']:.1f}C | Gas={sensor_state['gas']:.0f} | RF State={class_names.get(rf_label, 'N/A')} ({rf_conf*100:.0f}%)"
    cv2.putText(
        frame,
        info_text,
        (15, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 255, 0),
        2,
    )

    cv2.imshow("Multi-Sensor Early Warning Fusion", frame)
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

is_running = False
cap.release()
cv2.destroyAllWindows()

check
